# Pha T - Nhịp 1: Xác thực HybridTEEstimator (Threshold = 50)


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
tables_dir = BASE / 'results' / 'tables'


In [8]:
def evaluate_hybrid_on_existing_results(csv_name, metric='variance', threshold=50):
    df = pd.read_csv(tables_dir / csv_name)
    df_ksg = df[df['estimator'] == 'KSG'].copy()
    df_amortized = df[df['estimator'].str.startswith('Amortized')].copy()
    df_amortized['estimator'] = 'Amortized'
    
    group_cols = ['config_name', 'coupling_c', 'noise_std', 'n_samples']
    group_cols = [c for c in group_cols if c in df.columns]
    
    h_ksg = df_ksg[df_ksg['n_samples'] < threshold].copy()
    h_amo = df_amortized[df_amortized['n_samples'] >= threshold].copy()
    
    h_ksg['estimator'] = 'Hybrid'
    h_amo['estimator'] = 'Hybrid'
    
    hybrid_rows = pd.concat([h_ksg, h_amo], ignore_index=True)
    df_all = pd.concat([df, hybrid_rows], ignore_index=True)
    return df_all, group_cols

def compare_hybrid(df_all, group_cols, dataset_name):
    print(f"\n{'='*50}\nDataset: {dataset_name}\n{'='*50}")
    df_agg = df_all.groupby(['n_samples', 'estimator'])['variance'].mean().unstack()
    print("Mean Variance by N:")
    print(df_agg)
    
    has_bug = False
    for n, row in df_agg.iterrows():
        if 'Hybrid' not in row or pd.isna(row['Hybrid']):
            continue
        hyb = row['Hybrid']
        ksg = row.get('KSG', np.inf)
        amo = row.get('Amortized', np.inf)
        if hyb > max(ksg, amo) + 1e-9:
            has_bug = True
            print(f"BUG DETECTED at N={n}: Hybrid={hyb}, KSG={ksg}, Amortized={amo}")
    
    if not has_bug:
        print("\nSUCCESS: Hybrid is never strictly worse than BOTH KSG and Amortized at any N.")
    
    return df_all


In [9]:
df_lin, g_lin = evaluate_hybrid_on_existing_results('phase_r_grid_summary.csv')
df_lin = compare_hybrid(df_lin, g_lin, 'Linear VAR')

df_per, g_per = evaluate_hybrid_on_existing_results('phase_r2_periodic_grid_summary.csv')
df_per = compare_hybrid(df_per, g_per, 'Periodic Coupling')

df_lin['dataset'] = 'linear'
df_per['dataset'] = 'periodic'
df_final = pd.concat([df_lin, df_per], ignore_index=True)
df_final.to_csv(tables_dir / 'phase_t_hybrid_comparison.csv', index=False)
print("\nSaved phase_t_hybrid_comparison.csv")



Dataset: Linear VAR
Mean Variance by N:
estimator  Amortized   Binning    Hybrid       KSG  Symbolic
n_samples                                                   
10          0.011680  0.023339  0.003723  0.003723  0.021977
20          0.006386  0.016417  0.005345  0.005345  0.016661
30          0.004208  0.015014  0.005271  0.005271  0.012507
50          0.002593  0.014069  0.002593  0.004430  0.007086
100         0.001347  0.009252  0.001347  0.002870  0.002559
200         0.000673  0.004235  0.000673  0.001750  0.001023

SUCCESS: Hybrid is never strictly worse than BOTH KSG and Amortized at any N.

Dataset: Periodic Coupling
Mean Variance by N:
estimator  Amortized   Binning    Hybrid       KSG  Symbolic
n_samples                                                   
10          0.017570  0.022894  0.003868  0.003868  0.016440
20          0.011050  0.016039  0.005410  0.005410  0.011942
30          0.007168  0.012430  0.006363  0.006363  0.009148
50          0.004548  0.008019  0.00454